# UdaPlay 02 Solution Project

## Purpose

This notebook provides the second corrected UdaPlay project demo.

It extends the memory demonstration by:

1. Using the course-provided `lib.agents.Agent` wrapper.
2. Running multiple queries in one shared session.
3. Demonstrating cross-turn references using **"that game"**.
4. Demonstrating web-search fallback for a current/latest information query.
5. Printing session history when the submitted `Agent` implementation exposes `get_session_runs()`.


### 1. Imports


In [10]:
from dotenv import load_dotenv
import os

load_dotenv()

from lib.udaplay_vector_store import UdaPlayVectorStore
from lib.udaplay_tools import ( 
    configure_udaplay_tools,
    retrieve_game,
    evaluate_retrieval,
    game_web_search,
)
from lib.agents import Agent
import lib.llm

In [3]:
pip install sentence_transformers

97.15s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 2. Vector Store

In [11]:
vector_store = UdaPlayVectorStore(
    persist_dir="./chroma_db/udaplay_games",
    collection_name="udaplay_games",
    reset_collection=False,
)

print("Vector store initialized.")


Vector store initialized.


### 3.Agent Creation

In [12]:
agent = Agent(
    model_name="gpt-4o-mini",
    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search,
    ],
    instructions="""
You are UdaPlay, a helpful video game assistant.
- Use retrieve_game first for questions about video game facts that may be in the local dataset.
- Use evaluate_retrieval to decide whether retrieved context is sufficient.
- Use game_web_search only when retrieval is insufficient or when the question asks for latest, current, recent, ongoing, or external information.
- Maintain memory for all turns that share the same session_id.
- Resolve pronouns and cross-turn references such as "it", "that game", "this game", "the title", and "that one".
- Do not treat follow-up questions as standalone when a prior referenced game exists in the session.
- Return clear, structured answers and include citations or source URLs when web search is used.
"""
)

print("UdaPlay Agent created using lib.agents.Agent.")


UdaPlay Agent created using lib.agents.Agent.


### 4. Functon to print response

In [13]:

def print_agent_response(label, response):
    print("=" * 100)
    print(label)
    print("=" * 100)

    final_state = response.get_final_state()

    messages = final_state.get("messages", [])
    total_tokens = final_state.get("total_tokens", 0)

    print("\nAnswer:")
    if messages:
        print(messages[-1].content)
    else:
        print("No messages found.")

    print("\nTool Usage Trace:")
    tools_used = []

    for m in messages:
        tool_calls = getattr(m, "tool_calls", None)

        if tool_calls:
            for call in tool_calls:
                tool_name = call.function.name
                tools_used.append(tool_name)
                print(f"- AI requested tool: {tool_name}")
                print(f"  Arguments: {call.function.arguments}")

        if getattr(m, "role", None) == "tool":
            tool_name = getattr(m, "name", "unknown_tool")
            print(f"- Tool returned result from: {tool_name}")
            print(f"  Result preview: {m.content[:500]}")

    print("\nTools Used:")
    print(tools_used if tools_used else "No tools used in this run.")

    print("\nTotal Tokens:")
    print(total_tokens)

    print("\n")


### 5. Single Session 

In [14]:


session_id = "udaplay_full_demo_02"

queries = [
    "Who developed FIFA 21?",
    "What platform was it released on?",
    "When was God of War Ragnarok released?",
    "What platform was that game released on?",
    "What are the latest updates about GTA 6?",
]

responses = []

for i, query in enumerate(queries, start=1):
    response = agent.invoke(
        query,
        session_id=session_id
    )
    responses.append(response)

    print_agent_response(
        f"Query {i}: {query}",
        response
    )


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Query 1: Who developed FIFA 21?

Answer:
FIFA 21 was developed by **EA Sports**. The game was released on October 9, 2020, for various platforms including Microsoft Windows, Nintendo Switch, PlayStation 4, and Xbox One, with enhanced versions for the PlayStation 5 and Xbox Series X and S launched on December 3, 2020. 

For more detailed information, you can visit the [Wikipedia page for FIFA 21](https://en.wikipedia.org/wiki/FIFA_21).

Tool Usage Trace:
- AI requested tool: retrieve_game
  Arguments: {"query":"FIFA 21 developer"}
- Tool returned result fro

### Session log


In [17]:

if hasattr(agent, "get_session_runs"):
    session_runs = agent.get_session_runs(session_id=session_id)

    print("=" * 100)
    print("Session Log --- for session ID - "+ session_id)
    
    print("=" * 100)

    for i, run in enumerate(session_runs, start=1):
        print(f"\nRun {i}:")
        print(run)

else:
    print("This Agent implementation does not expose get_session_runs().")


Session Log --- for session ID - udaplay_full_demo_02

Run 1:
Run('7b95f795-ad79-4c3e-ae7f-24988cf110d4')

Run 2:
Run('714ab2cf-cc91-412c-978f-1479a19d610d')

Run 3:
Run('5fdb4793-9cc5-4dd2-993a-81b9c8591bf4')

Run 4:
Run('39cef03e-1834-42f4-8f3f-98ee4cc4edd4')

Run 5:
Run('202d7d54-3a46-4943-8bbd-1e3fa56763ab')


### Rubric Tracebility


- Use of the course `lib.agents.Agent` wrapper.
- Internal retrieval through `retrieve_game`.
- Retrieval evaluation through `evaluate_retrieval`.
- Web search fallback through `game_web_search`.
- Stateful conversation in a session `session_id`.
- Pronoun resolution using **"it"** and **"that game"**.
- Session Log output through `get_session_runs()`.
